In [3]:
%pip install ultralytics

import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from ultralytics import YOLO # pyright: ignore[reportMissingImports]

RAW_DIR = Path("/Users/maliya/Desktop/dissertation/Vista/data/raw/consolidated/vehicle-sign-detection")
PROCESSED_DIR = Path("/Users/maliya/Desktop/dissertation/Vista/data/processed/yolo_traffic_signs")
SPLITS = ["train", "valid", "test"]

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    return cx, cy, w / img_w, h / img_h


def convert_split(split, class_id_map):
    coco = json.load(open(RAW_DIR / split / "_annotations.coco.json"))
    images_by_id = {img["id"]: img for img in coco["images"]}

    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["category_id"] in class_id_map:
            anns_by_image[ann["image_id"]].append(ann)

    out_images = PROCESSED_DIR / split / "images"
    out_labels = PROCESSED_DIR / split / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    for image_id, img in images_by_id.items():
        src = RAW_DIR / split / img["file_name"]
        dst = out_images / img["file_name"]
        if not dst.exists():
            dst.symlink_to(src.resolve())

        lines = []
        for ann in anns_by_image.get(image_id, []):
            cx, cy, w, h = coco_bbox_to_yolo(ann["bbox"], img["width"], img["height"])
            lines.append(f"{class_id_map[ann['category_id']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        label_path = out_labels / f"{Path(img['file_name']).stem}.txt"
        label_path.write_text("\n".join(lines))

    return len(images_by_id), sum(len(v) for v in anns_by_image.values())

In [ ]:
train_coco = json.load(open(RAW_DIR / "train" / "_annotations.coco.json"))
sign_categories = sorted(
    (c for c in train_coco["categories"] if c["name"].startswith("s_")),
    key=lambda c: c["id"],
)
class_id_map = {c["id"]: i for i, c in enumerate(sign_categories)}
class_names = [c["name"] for c in sign_categories]
print(f"{len(class_names)} traffic-sign classes: {class_names}")

for split in SPLITS:
    n_images, n_boxes = convert_split(split, class_id_map)
    print(f"{split}: {n_images} images, {n_boxes} sign boxes")

## 2. Write the YOLO dataset config

Ultralytics reads dataset layout and class names from a `data.yaml` pointing at the `train`/`val`/`test` image folders we just created.

In [ ]:
data_yaml_path = PROCESSED_DIR / "data.yaml"
data_yaml_path.write_text(
    f"path: {PROCESSED_DIR}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    "names:\n"
    + "\n".join(f"  {i}: {name}" for i, name in enumerate(class_names))
    + "\n"
)
print(data_yaml_path.read_text())

## 3. Train YOLO

Fine-tune a pretrained YOLOv8n checkpoint on the traffic-sign classes. Ultralytics auto-selects the best available device (Apple Silicon MPS on this machine).

Note: only 588 of 5169 training images contain a sign box (891 boxes across 20 classes, heavily skewed toward `s_yield`/`s_voorrang`), so treat this as a proof-of-concept pipeline — expect weak recall on the rarer sign types (e.g. `s_deadend`, `s_warning_junction`) until more labeled data is added.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    project="runs/detect",
    name="traffic_sign_yolo",
)

## 4. Validate on the held-out test split

In [ ]:
metrics = model.val(data=str(data_yaml_path), split="test")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 5. Run inference on sample test images

In [ ]:
test_images = sorted((PROCESSED_DIR / "test" / "images").glob("*.jpg"))[:6]
predictions = model.predict(source=[str(p) for p in test_images], conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pred in zip(axes.flat, predictions):
    ax.imshow(pred.plot()[..., ::-1])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Export to Core ML

Mirrors the deployment target used for the MobileNetV2 classifier notebook, so both models can ship on-device.

In [ ]:
export_path = model.export(format="coreml")
print(f"Exported to {export_path}")